QR DQN Ablation Study 1:

10M steps + **4 Parallel Envs** + 4 Actions

Logan Wong

law3082

In [ ]:
# REMINDER: make sure you set
# Runtime
# Change runtime type
# T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari
!ls -la

In [ ]:
!pip install gymnasium[atari,accept-rom-license] ale-py sb3_contrib stable-baselines3

# Install tensorboard
!pip install tensorboard

In [ ]:
# Load tensorboard extension
%load_ext tensorboard

# If you need to reload it
# %reload_ext tensorboard

In [ ]:
import os
import torch
import gymnasium as gym
import stable_baselines3
import ale_py
import numpy as np
import random

# Algorithm
from sb3_contrib import QRDQN


# For debugging
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.callbacks import BaseCallback
import time

# Action masking
from gymnasium import ActionWrapper
from stable_baselines3.common.atari_wrappers import AtariWrapper

# Vector environment
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv

print("All imports working")

In [ ]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
def convert(seconds):
    seconds = seconds % (24 * 3600)
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60

    return "%d:%02d:%02d" % (hour, minutes, seconds)

Create Environment and Model

In [ ]:
# Action space: Discrete(6)
# Number of actions: 6
# Action meanings: ['NOOP', 'FIRE', 'UP', 'DOWN', 'UPFIRE', 'DOWNFIRE']
# Observation shape: (210, 160, 3)
# Observation space: Box(0, 255, (210, 160, 3), uint8)

# ACTIONS:
# 0: NOOP
# 1: FIRE
# 2: UP
# 3: DOWN

# 4: UPFIRE
# 5: DOWNFIRE

class ActionReducer(ActionWrapper):
  def __init__(self, env):
    super().__init__(env)

    # NOOP, FIRE, UP, and DOWN only. No UPFIRE. No DOWNFIRE.
    self.allowed_actions = [0,1,2,3]

    self.action_space = gym.spaces.Discrete(len(self.allowed_actions))

  def action(self, action):
    return self.allowed_actions[action]

In [ ]:
seed = 5000
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
game_name = "ALE/Bowling-v5"
n_envs = 4

# make_atari_env internally uses make_vec_env
# wrapper_kwargs passes clip_reward to the AtariWrapper
env = make_atari_env(
    game_name,
    n_envs=n_envs,         # Creates parallel envs that run simultaneously
    seed=seed,
    wrapper_kwargs=dict(clip_reward=False)
)

# n_stack gives 4 consecutive frames as input for each env
env = VecFrameStack(env, n_stack=4)

env.action_space.seed(seed)

In [ ]:
tensorboard_save_path = "/content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/code/qr_dqn_bowling_tensorboard/QRDQN_Ablation_Study_1"

model = QRDQN(
    "CnnPolicy",
    env,
    learning_rate=0.0001,
    buffer_size=50000,
    batch_size=32,
    gamma=0.99,
    target_update_interval=1000,
    train_freq=4//n_envs,
    gradient_steps=1,
    exploration_final_eps=0.02,
    exploration_fraction=0.1,
    learning_starts=10000,

    seed=seed,
    verbose=1,
    device="cuda",
    tensorboard_log=tensorboard_save_path
)

print("QR-DQN model created")

Train the model

In [ ]:
total_timesteps = 10000000    # 10M

# Time how long it takes
print("Training started")
start_time = time.time()

# log_interval: Print train metrics every 10 episodes
model.learn(
    total_timesteps=total_timesteps,
    progress_bar=True,
    log_interval=10
)
end_time = time.time()
print("Training done")

env.close()

# Calculate run time
training_duration = end_time - start_time
time_in_minutes_and_seconds = convert(training_duration)
print(f"Time taken: {time_in_minutes_and_seconds}")

# Save model to Google Drive
trained_model_save_path = f"/content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/models/qr_dqn_{total_timesteps}_Ablation_Study_1"
model.save(trained_model_save_path)
print("Model saved to Google Drive")

In [ ]:
# Kill any existing TensorBoard
# !pkill -f tensorboard

# Activate Tensorboard
%tensorboard --logdir /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/code/qr_dqn_bowling_tensorboard/

# Ablation Study 1 is QRDQN_Ablation_Study_1

In [ ]:
torch.cuda.empty_cache()
del model
del env

In [ ]:
# Total timestesp: 10M
# Time taken:
# Final Mean reward: